# EDA — Khám Phá Dữ Liệu

Notebook này chỉ tập trung vào **EDA (Exploratory Data Analysis)** — quan sát cấu trúc, phân phối và chất lượng dữ liệu **thô**, chưa làm sạch.

**Datasets:**
- `log_standard_4_22_to_5_08_1k.csv` — log chính (main period)
- `log_standard_4_08_to_4_21_1k.csv` — log trước đó (train ML)
- `log_random_4_22_to_5_08_1k.csv` — log random
- `user_features_1k.csv` — đặc trưng người dùng
- `video_features_basic_1k.csv` — đặc trưng video cơ bản

## 0. Import & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 4)

DATA_DIR = "../data/raw/"

print("Đang load data...")
log_main  = pd.read_csv(DATA_DIR + "log_standard_4_22_to_5_08_1k.csv")
log_prev  = pd.read_csv(DATA_DIR + "log_standard_4_08_to_4_21_1k.csv")
log_rand  = pd.read_csv(DATA_DIR + "log_random_4_22_to_5_08_1k.csv")
user_feat = pd.read_csv(DATA_DIR + "user_features_1k.csv")
vid_basic = pd.read_csv(DATA_DIR + "video_features_basic_1k.csv")

print(f" log_main  : {len(log_main):>10,} rows")
print(f" log_prev  : {len(log_prev):>10,} rows")
print(f" log_rand  : {len(log_rand):>10,} rows")
print(f" user_feat : {len(user_feat):>10,} rows")
print(f" vid_basic : {len(vid_basic):>10,} rows")

---
## 1. EDA — `log_main` (log_standard_4_22_to_5_08)

### 1.1 Cấu trúc cơ bản

In [ ]:
print("Shape:", log_main.shape)
log_main.dtypes

In [ ]:
log_main.head()

In [ ]:
log_main.describe(include='all')

### 1.2 Kiểm tra Null & Duplicate

In [ ]:
print("=== Null Values ===")
null_info = log_main.isnull().sum()
print(null_info[null_info > 0].to_string() or "Không có null")

print(f"\n=== Duplicates ===")
n_dup = log_main.duplicated().sum()
print(f"Số dòng trùng: {n_dup:,} ({n_dup/len(log_main)*100:.2f}%)")

### 1.3 Phân phối `play_time_ms` và `duration_ms`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

log_main['play_time_ms'].clip(upper=log_main['play_time_ms'].quantile(0.99)).hist(
    bins=50, ax=axes[0], color='steelblue', edgecolor='white'
)
axes[0].set_title('play_time_ms (clip 99th percentile)')
axes[0].set_xlabel('ms')

log_main['duration_ms'].clip(upper=log_main['duration_ms'].quantile(0.99)).hist(
    bins=50, ax=axes[1], color='coral', edgecolor='white'
)
axes[1].set_title('duration_ms (clip 99th percentile)')
axes[1].set_xlabel('ms')

plt.tight_layout()
plt.show()

### 1.4 Watch Ratio (thô)

In [ ]:
# Tính watch_ratio thô để quan sát, không lưu vào df
watch_ratio_raw = (
    log_main['play_time_ms'] / log_main['duration_ms'].replace(0, np.nan)
)

print(watch_ratio_raw.describe())
print(f"\nwatch_ratio > 1 (play > duration): {(watch_ratio_raw > 1).sum():,} rows")
print(f"watch_ratio = NaN (duration = 0) : {watch_ratio_raw.isnull().sum():,} rows")

watch_ratio_raw.clip(0, 2).hist(bins=60, color='mediumseagreen', edgecolor='white')
plt.title('Phân phối Watch Ratio (thô, clip 0–2)')
plt.xlabel('watch_ratio')
plt.axvline(1.0, color='red', linestyle='--', label='ratio = 1')
plt.legend()
plt.show()

### 1.5 Tương tác người dùng (clicks, likes, shares)

In [ ]:
interaction_cols = ['is_click', 'is_like', 'is_forward']
available = [c for c in interaction_cols if c in log_main.columns]

rates = log_main[available].mean() * 100
print("Tỷ lệ tương tác (%)")
print(rates.round(2).to_string())

rates.plot(kind='bar', color=['#4C8BE0', '#E06C4C', '#4CE08B'], edgecolor='white', rot=0)
plt.title('Tỷ lệ tương tác (%) trong log_main')
plt.ylabel('%')
plt.tight_layout()
plt.show()

### 1.6 Phân phối theo thời gian

In [ ]:
event_time = pd.to_datetime(log_main['time_ms'], unit='ms')
print(f"Khoảng thời gian: {event_time.min()} → {event_time.max()}")

event_time.dt.date.value_counts().sort_index().plot(
    kind='bar', figsize=(14, 4), color='slateblue', edgecolor='white'
)
plt.title('Số events theo ngày — log_main')
plt.xlabel('Ngày')
plt.ylabel('Số events')
plt.tight_layout()
plt.show()

---
## 2. EDA — `user_features`

### 2.1 Cấu trúc & Null

In [ ]:
print("Shape:", user_feat.shape)
user_feat.head()

In [ ]:
null_user = user_feat.isnull().sum()
null_user = null_user[null_user > 0]
print("Null values:")
print(null_user.to_string() if len(null_user) else "Không có null")

### 2.2 Phân phối `user_active_degree`

In [ ]:
if 'user_active_degree' in user_feat.columns:
    vc = user_feat['user_active_degree'].value_counts()
    print(vc)
    vc.plot(kind='bar', color='teal', edgecolor='white', rot=30)
    plt.title('Phân phối user_active_degree')
    plt.tight_layout()
    plt.show()

### 2.3 Thống kê các cột số

In [ ]:
user_feat.describe(include='number')

---
## 3. EDA — `video_features_basic`

### 3.1 Cấu trúc & Null

In [ ]:
print("Shape:", vid_basic.shape)
vid_basic.head()

In [ ]:
null_vid = vid_basic.isnull().sum()
null_vid = null_vid[null_vid > 0]
print("Null values:")
print(null_vid.to_string() if len(null_vid) else "Không có null")

### 3.2 Phân phối `video_type`

In [ ]:
if 'video_type' in vid_basic.columns:
    vc = vid_basic['video_type'].value_counts()
    print(vc)
    vc.plot(kind='bar', color='salmon', edgecolor='white', rot=0)
    plt.title('Phân phối video_type')
    plt.tight_layout()
    plt.show()

### 3.3 Phân phối `video_duration` (ms)

In [ ]:
if 'video_duration' in vid_basic.columns:
    dur = vid_basic['video_duration'].dropna()
    print(dur.describe())
    dur.clip(upper=dur.quantile(0.99)).hist(bins=50, color='orchid', edgecolor='white')
    plt.title('Phân phối video_duration (ms, clip 99th)')
    plt.xlabel('ms')
    plt.tight_layout()
    plt.show()

---
## 4. Tổng Kết EDA

| Dataset | Rows | Duplicates | Nulls đáng chú ý |
|---|---|---|---|
| log_main | – | Cần kiểm tra | – |
| log_prev | – | Cần kiểm tra | – |
| user_feat | – | – | user_active_degree, cột số |
| vid_basic | – | – | tag, upload_dt |

> Điền kết quả thực tế sau khi chạy các cell trên.

**Các điểm cần xử lý ở bước preprocess:**
- `log_main` / `log_prev`: drop duplicate, tính `watch_ratio`, chuyển `time_ms` → datetime
- `user_feat`: fill null số bằng median, fill null string bằng `'UNKNOWN'`, đánh dấu `is_low_quality`
- `vid_basic`: lọc bỏ video không phải `NORMAL`, chuyển duration ms → giây, fill null tag/upload_dt

---
## 5. Export Sample Files

Chia `log_main` thành các file sample với kích thước khác nhau phục vụ các mục đích khác nhau.

| File | Dòng | Size (ước tính) | Dùng cho |
|------|------|-----------------|----------|
| `kuairand_logs_100k.csv` | 100,000 | ~6.7 MB | Test nhanh |
| `kuairand_logs_500k.csv` | 500,000 | ~33.4 MB | Demo |
| `kuairand_logs_1000k.csv` | 1,000,000 | ~66.7 MB | Benchmark |
| `kuairand_logs_full_clean.csv` | full | ~440 MB | Production |

In [ ]:
import os

SAMPLE_DIR = "../data/sample/"
os.makedirs(SAMPLE_DIR, exist_ok=True)

# Chuẩn bị df export: chuyển time_ms → string ISO để tương thích JSON/Kafka
df_export = log_main.copy()
df_export['event_time'] = pd.to_datetime(df_export['time_ms'], unit='ms').astype(str)

total_rows = len(df_export)
print(f"Tổng rows log_main: {total_rows:,}")
print()

In [ ]:
# ── Sample files ──────────────────────────────────────────────────────────────
samples = {
    '100k'  :   100_000,
    '500k'  :   500_000,
    '1000k' : 1_000_000,
}

print('=' * 60)
print(f"{'File':<45} {'Dòng':>10}  {'Size':>8}")
print('=' * 60)

for name, n in samples.items():
    if n > total_rows:
        print(f"  ⚠️  Bỏ qua '{name}': yêu cầu {n:,} > có {total_rows:,} rows")
        continue
    sample = df_export.sample(n=n, random_state=42)
    path   = SAMPLE_DIR + f"kuairand_logs_{name}.csv"
    sample.to_csv(path, index=False)
    size_mb = os.path.getsize(path) / 1e6
    print(f"  {path:<43} {n:>10,}  {size_mb:>6.1f} MB")

# ── Full clean export ─────────────────────────────────────────────────────────
full_path = SAMPLE_DIR + "kuairand_logs_full_clean.csv"
df_export.to_csv(full_path, index=False)
size_full = os.path.getsize(full_path) / 1e6
print(f"  {full_path:<43} {total_rows:>10,}  {size_full:>6.1f} MB")

print('=' * 60)
print(f"\n✅ Xong! Kiểm tra thư mục: {SAMPLE_DIR}")